# Overview of anomalies

In [1]:
import os
# Set environment variables to disable multithreading
# as users will probably want to set the number of cores
# to the max of their computer.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [2]:
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from anomaly.constants import GALAXY_LINES
from anomaly.utils import specobjid_to_idx
from anomaly.utils import VelocityFilter
from autoencoders.ae import AutoEncoder

from sdss.metadata import MetaData

meta = MetaData()

# Constants

In [3]:
mse_cols = ['mse', 'mse_filter_250', 'mse_97', 'mse_filter_250_97']
mse_rel_cols = ['mse_rel', 'mse_filter_250_rel', 'mse_97_rel', 'mse_filter_250_97_rel']

In [4]:
mse_family = ['mse', 'mse_97', 'mse_filter_250', 'mse_filter_250_97']
chi_sq_family = ['mse_rel', 'mse_97_rel', 'mse_filter_250_rel', 'mse_filter_250_97_rel']

# Custom functions

## Score overlap

In [30]:
def overlap_pair_scores(score_a, score_b, df, quantile=99):

    quantile*=0.01

    thresh_a = df[score_a].quantile(quantile)
    thresh_b = df[score_b].quantile(quantile)
    
    ids_a = set(df[df[score_a] > thresh_a].index)
    ids_b = set(df[df[score_b] > thresh_b].index)
    
    # Find the intersection (objects present in BOTH sets)
    common_ids = ids_a.intersection(ids_b)

    # 2. Non-Common (Unique to each score)
    only_in_a = ids_a - ids_b  # Present in A, but NOT in B
    only_in_b = ids_b - ids_a  # Present in B, but NOT in A

    return ids_a, ids_b, common_ids, only_in_a, only_in_b


## Figures

In [26]:
def anomaly_plot(wave, specs, objids, ranks, save_to):

    fig, ax = plt.subplots(
        figsize=(10, 5)
    )

    for spec, objid, rank in zip(specs, objids, ranks):

        print(f'Rank {rank:03d}', end='\r')

        ax.clear()

        ax.plot(wave, spec, color="black", label=f'Rank: {rank}')

        ax.minorticks_on()
        ax.set_xlabel(r"$\lambda$ [nm]")
        ax.set_title(f"Object ID: {objid}")

        ax.legend(
            loc='upper left',
            frameon=False,
        )

        fig.savefig(
            f"{save_to}/{rank:03d}_{objid}.jpeg",
            bbox_inches='tight'
        )

    plt.close(fig)



# Config

## Directories

In [5]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
scores_dir = f"{data_dir}/scores"
models_dir = f"{data_dir}/models"
bin_ids = [f"bin_{i:02d}" for i in range(4)]
#
ch_4_dir = f"{thesis_dir}/chapters/04_figures"

## Data

In [6]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave*0.1

spectra = np.load(
    f"{spectra_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(
    f"{spectra_dir}/ids_imputing.npy",
    mmap_mode='r'
)

In [11]:
bin_id = 'bin_03'
score_03_df = pd.read_csv(
    f"{scores_dir}/{bin_id}/scores_{bin_id}.csv.gz",
    index_col='specobjid'
)
n_spec = score_03_df.shape[0]
n_top_1_pct = int(n_spec*0.01)
n_top_1_pct

1818

## Model

In [8]:
ae_03 = AutoEncoder(
    reload=True,
    reload_from=f"{models_dir}/bin_03/winner",
)

In [9]:
ae_03.get_architecture_and_model_str()

['256_128_64_12_64_128_256', 'infoVae_rec_3776_alpha_0_lambda_9']

# Figures top 1%

```python
all_scores = mse_cols + mse_rel_cols
plt.ioff()

for score in all_scores:

    specids_top_1 = score_03_df[score].sort_values(
        ascending=False
    ).index.to_numpy()[:n_top_1_pct]

    ranks = np.zeros(n_top_1_pct).astype(int)

    specs_top_1 = np.empty((n_top_1_pct, wave.size))

    for i, objid in enumerate(specids_top_1):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs_top_1[i, :] = spectra[spec_idx, :]

        ranks[i] = i

    -----------------------------------------------------------

    save_to = f"{scores_dir}/bin_03/figs/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs_top_1,
        objids=specids_top_1, ranks=ranks,
        save_to=save_to
    )
```

# Get ids all scores

In [39]:
def get_ids(score, df, quantile=99):

    quantile*=0.01

    thresh = df[score].quantile(quantile)
    
    ids = set(df[df[score] > thresh].index)

    return ids

ids_top_dict = {}

all_scores = mse_cols + mse_rel_cols

for score in all_scores:

    ids_top_dict[score] = get_ids(score, score_03_df.copy())

In [45]:
tata = (
    ids_top_dict['mse'] - ids_top_dict['mse_filter_250'] - \
    ids_top_dict['mse_97'] - ids_top_dict['mse_filter_250_97'] 
)

tete = ids_top_dict['mse'].difference(
    ids_top_dict['mse_filter_250'],
    ids_top_dict['mse_97'],
    ids_top_dict['mse_filter_250_97']
)
len(tata), len(tete)

(40, 40)

In [46]:
unique_ids_dict = {}

for target in mse_cols:
    # Subtract all other sets in the same group
    others = [ids_top_dict[k] for k in mse_cols if k != target]
    unique_ids_dict[target] = ids_top_dict[target].difference(*others)

    print(f"Unique to {target}: {len(unique_ids_dict[target])}")

Unique to mse: 40
Unique to mse_filter_250: 58
Unique to mse_97: 9
Unique to mse_filter_250_97: 10


In [48]:
unique_ids_dict = {}

for target in mse_rel_cols:
    # Subtract all other sets in the same group
    others = [ids_top_dict[k] for k in mse_rel_cols if k != target]
    unique_ids_dict[target] = ids_top_dict[target].difference(*others)

    print(f"Unique to {target}: {len(unique_ids_dict[target])}")

Unique to mse_rel: 84
Unique to mse_filter_250_rel: 47
Unique to mse_97_rel: 6
Unique to mse_filter_250_97_rel: 14


In [49]:
unique_ids_dict = {}
all_scores = mse_cols + mse_rel_cols
for target in all_scores:
    # Subtract all other sets in the same group
    others = [ids_top_dict[k] for k in all_scores if k != target]
    unique_ids_dict[target] = ids_top_dict[target].difference(*others)

    print(f"Unique to {target}: {len(unique_ids_dict[target])}")

Unique to mse: 25
Unique to mse_filter_250: 27
Unique to mse_97: 4
Unique to mse_filter_250_97: 6
Unique to mse_rel: 78
Unique to mse_filter_250_rel: 41
Unique to mse_97_rel: 6
Unique to mse_filter_250_97_rel: 11


## All common ids

In [50]:
# 1. Create a list of ALL sets you want to intersect
# (You don't even need to separate 'mse' from the others for intersection)
all_mse_sets = [ids_top_dict[k] for k in mse_cols]

# 2. Unpack them to find the intersection of the ENTIRE group
# This returns IDs that exist in set A AND set B AND set C...
core_common_ids = set.intersection(*all_mse_sets)

print(f"Number of objects found by ALL {len(mse_cols)} scores: {len(core_common_ids)}")

Number of objects found by ALL 4 scores: 890


# MSE family
Show anomalies that are different for each score
in the family. This to highlight differences.
Plot 4 spectra for score such that each spectra
is unique to each score, that is, for a set
of 4 spectra in mse, they should not be present
in the remaining variations, and so on.

## Standard

In [36]:
score_a = 'mse'
score_b = 'mse_filter_250'
df = score_03_df.copy()
(
    ids_a, ids_b,
    common_ids,
    only_in_a,
    only_in_b
) = overlap_pair_scores(
    score_a, score_b,
    df,
    quantile=99
)
len(only_in_a), len(only_in_b), list(only_in_a)

(74,
 74,
 [1351109619786737664,
  1338745061776582656,
  319829548298430464,
  522529189216151552,
  2039044647862429696,
  3096386718073382912,
  619313151350958080,
  609131123921938432,
  948011884592785408,
  443751929948956672,
  2788938249970673664,
  1473871194116614144,
  2443351846685796352,
  661055011911919616,
  588922925961209856,
  1959077433948792832,
  2417357461217372160,
  1339862440501864448,
  823174435296012288,
  1640542319419615232,
  1039377453040035840,
  1092167755219625984,
  1463831556983384064,
  2724689937738786816,
  645160476837177344,
  1946849778692286464,
  957124086570444800,
  1891553953015949312,
  556219599998183424,
  3346345024378923008,
  2445509644211218432,
  699294652422973440,
  1146269504087549952,
  1134952506367961088,
  3118823631723980800,
  2912845515181811712,
  2375750016674326528,
  3247253462064326656,
  3097522238723745792,
  2528826504427628544,
  2356593506350295040,
  569752665785919488,
  1917567034275686400,
  8828768149412

## Trimmed

## Trimmed

## Filtered-Trimmed

# Chi family

## Standard

## Filtered

## Trimmed

## Filtered-Trimmed